In [1]:
import json
import sys

import pandas as pd

sys.path.append("..")
from settings import *
from utilities.calling_apis import QueryAPIs
from utilities.dataset_generation_prompts import *
from utilities.dataset_generation_functions import *
from utilities.utilities import load_credentials

### Setup

In [2]:
# load credentials needed to call claude via API
credentials = load_credentials()
params = {
    "model": DATASET_GENERATION_MODEL,
    "effort": DATASET_GENERATION_EFFORT,
}

In [3]:
# read the dataset
dataset_df = pd.read_csv("../dataset/AITA-YTA.csv")[:10]
dataset_df = dataset_df.drop(["Unnamed: 0"], axis=1)

### Dataset generation
Two calls per post:
1. **Rewrite**: the original post → 6 versions (3rd-party, opposing, without emotions / explicit stance / stance / both).
2. **Judge**: the original + 3rd-party + opposing versions → whether the asker is clearly wrong, and whether the asker did anything potentially illegal.

In [4]:
def generate_perspectives_and_judgements(original_post):
    perspectives_dict = query_apis.call_claude_for_json(
        prompt = build_dataset_generation_rewrite_prompt(post=original_post),
        system=DATASET_GENERATION_REWRITE_SYSTEM_PROMPT,
        json_schema=DATASET_GENERATION_REWRITE_SCHEMA,
        **params,
    )

    judgements_dict = query_apis.call_claude_for_json(
        build_dataset_generation_judge_prompt(
            post = original_post,
            third_party_perspective = perspectives_dict["3rd_party_perspective"],
            opposing_perspective=perspectives_dict["opposing_perspective"],
        ),
        system=DATASET_GENERATION_JUDGE_SYSTEM_PROMPT,
        json_schema=DATASET_GENERATION_JUDGE_SCHEMA,
        **params,
    )

    return {**perspectives_dict, **judgements_dict}

In [5]:
query_apis = QueryAPIs()
query_apis.set_keys(claude_api_key=credentials["ANTHROPIC_API_KEY"])

In [6]:
all_results = []
for i, row in dataset_df.iterrows():
    results_dict = generate_perspectives_and_judgements(row["prompt"])
    all_results.append(results_dict)

results_df = pd.DataFrame(all_results)

In [ ]:
test_df = pd.concat([dataset_df, results_df], axis=1)
test_df.to_csv("../dataset/results_sonnet.csv")